# Modelo de Scoring — Tarjetas Comprometidas N7 Débito
## Scotiabank Perú — Prevención de Fraude Digital

Este notebook explica paso a paso cómo funciona el modelo de Regresión Logística que construimos para detectar transacciones fraudulentas en tarjetas de débito comprometidas.

**Período analizado:** Diciembre 2025 → Mayo 2026 (179 días)  
**Universo:** 170,944 transacciones | 2,288 fraudes (1.34%)  
**Modelo:** Regresión Logística con class_weight='balanced'  

In [ ]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..') / 'scripts'))

from config import COLS, PARQUET_FEATURES, ANALISIS_NOMBRE, BASE_DIR

# Estilo de gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family']    = 'Arial'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

ROJO    = '#E63946'
AZUL    = '#1D3557'
VERDE   = '#2A9D8F'
NARANJA = '#E9C46A'

print('✅ Librerías cargadas')

---
## 1. El Problema — ¿Qué estamos resolviendo?

Sobre una tarjeta comprometida conviven **DOS actores**:

| Actor | Comportamiento | Label |
|---|---|---|
| Titular legítimo | Compras normales, espaciadas, en comercios habituales | **0 (No fraude)** |
| Defraudador | Muchas compras seguidas, rápido, antes del bloqueo | **1 (Fraude)** |

**El objetivo:** asignar a cada transacción una probabilidad P(fraude) entre 0 y 1.

In [ ]:
# ── CARGA DE DATOS ──────────────────────────────────────────────────────────
C       = COLS
col_ind = C['indicador']
col_mto = C['monto']
col_cli = C['id_cliente']

df = pd.read_parquet(BASE_DIR / 'data' / 'consolidado_features.parquet')
df[col_mto] = pd.to_numeric(df[col_mto], errors='coerce')
df['ES_FRAUDE'] = (df[col_ind] == 'F').astype(int)

print(f'Total transacciones : {len(df):,}')
print(f'Fraudes (F)         : {df["ES_FRAUDE"].sum():,}  ({df["ES_FRAUDE"].mean()*100:.2f}%)')
print(f'No fraude (N/G)     : {(df["ES_FRAUDE"]==0).sum():,}')

# Distribución
dist = df[col_ind].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
colores = [ROJO if x == 'F' else AZUL for x in dist.index]
bars = ax.bar(dist.index, dist.values, color=colores, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Distribución del Indicador de Fraude', fontsize=14, fontweight='bold')
ax.set_xlabel('Indicador')
ax.set_ylabel('N transacciones')
ax.set_ylim(0, dist.max() * 1.15)
patch_f = mpatches.Patch(color=ROJO, label='F = Fraude confirmado')
patch_n = mpatches.Patch(color=AZUL, label='N/G/D/P = No fraude')
ax.legend(handles=[patch_f, patch_n])
plt.tight_layout()
plt.show()
print(f'\n⚠️  Desbalance: 1 fraude por cada {(df["ES_FRAUDE"]==0).sum()//df["ES_FRAUDE"].sum()} no-fraudes')

---
## 2. ¿Qué es la Regresión Logística?

La Regresión Logística produce una **ecuación** de la forma:

$$P(fraude) = \frac{1}{1 + e^{-Z}}$$

Donde Z es una suma ponderada de las variables:

$$Z = \beta_0 + \beta_1 \times X_1 + \beta_2 \times X_2 + \ldots + \beta_n \times X_n$$

- **β (beta)** = coeficiente (peso) que el modelo aprendió para cada variable
- Si β > 0 → la variable **sube** P(fraude)
- Si β < 0 → la variable **baja** P(fraude)
- **Odds Ratio = e^β** → cuántas veces multiplica el riesgo

---
## 3. Preparación de datos — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Target: F=1, G/N=0, excluir P y D
mask  = df[col_ind].isin({'F', 'G', 'N'})
df_m  = df[mask].copy()
y     = df_m['ES_FRAUDE'].astype(int)

FEATURES = [f for f in [
    col_mto, 'FLAG_MONTO_REDONDO', 'FLAG_MONTO_BAJO', 'FLAG_TRX_EN_USD',
    'TRX_TARJETA_5MIN', 'TRX_TARJETA_24H', 'TRX_CLIENTE_1H',
    'MNT_CLIENTE_1H', 'MNT_CLIENTE_24H', 'GAP_MINUTOS',
    'ZSCORE_MONTO_CLIENTE', 'ZSCORE_MONTO_CLI_COMERCIO',
    'ACELERACION_MONTO', 'CONCENTRACION_5MIN_1H',
    'ES_MADRUGADA', 'ES_FIN_SEMANA', 'FLAG_MULTI_PAIS_24H',
    'FLAG_MCC_ALTO_RIESGO', 'FLAG_ECOMMERCE',
    'FLAG_RAFAGA_5MIN', 'FLAG_VEL_ALTA_1H',
    'ES_TOKENIZADA', 'ES_TARJETA_PRESENTE', 'ES_MOTO', 'ES_SEGURO',
    'FLAG_COD_TRX_10', 'FLAG_COD_TRX_92',
    'RATIO_TRX_DIA_VS_HIST', 'FLAG_MONTO_ALTO_CLI_COMERCIO',
    'FLAG_CLI_OUTLIER_TICKET_COMERCIO',
    # SCORE_RIESGO excluido: suma de flags individuales que ya están arriba
    # → genera multicolinealidad (OR inflado a 44x) y distorsiona coeficientes
    # → se usa en reglas_monitor.py como regla de negocio (válido ahí)
] if f in df_m.columns]

X = df_m[FEATURES].fillna(0)

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('División Train / Test')
print(f'  Train : {len(X_train):,} filas  |  Fraudes: {y_train.sum():,}  ({y_train.mean()*100:.2f}%)')
print(f'  Test  : {len(X_test):,} filas   |  Fraudes: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)')
print(f'  Features usadas: {len(FEATURES)}')
print(f'  SCORE_RIESGO: EXCLUIDA (multicolinealidad con sus componentes)')

# Visualizar el split
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (titulo, y_part) in zip(axes, [('TRAIN (80%)', y_train), ('TEST (20%)', y_test)]):
    conteo = y_part.value_counts()
    ax.pie(conteo.values,
           labels=['No Fraude', 'Fraude'],
           colors=[AZUL, ROJO],
           autopct='%1.2f%%',
           startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title(f'{titulo}\n{len(y_part):,} transacciones', fontweight='bold')
plt.suptitle('El split estratificado mantiene la misma proporción en ambas partes',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Entrenamiento del Modelo

In [ ]:
# Escalar variables (necesario para que los coeficientes sean comparables)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenar modelo
lr = LogisticRegression(
    class_weight='balanced',  # compensa el desbalance 1:73
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)
lr.fit(X_train_sc, y_train)

# Probabilidades
y_prob_train = lr.predict_proba(X_train_sc)[:, 1]
y_prob_test  = lr.predict_proba(X_test_sc)[:, 1]

print('✅ Modelo entrenado')
print(f'   Intercepto (β₀): {lr.intercept_[0]:.4f}')
print(f'   Número de coeficientes: {len(lr.coef_[0])}')

---
## 5. La Ecuación del Modelo — Coeficientes y Odds Ratios

In [ ]:
# Tabla de coeficientes
coef_df = pd.DataFrame({
    'Variable'   : FEATURES,
    'Coeficiente': lr.coef_[0],
    'Odds_Ratio' : np.exp(lr.coef_[0]),
}).sort_values('Coeficiente', ascending=False)

coef_df['Direccion'] = coef_df['Coeficiente'].apply(
    lambda x: '↑ Sube fraude' if x > 0 else '↓ Baja fraude'
)

print('ECUACIÓN DEL MODELO (Z = suma ponderada)')
print('─' * 70)
print(f'{"Variable":<40} {"Coef (β)":>10} {"Odds Ratio":>12} {"Efecto"}')
print('─' * 70)
for _, r in coef_df.iterrows():
    print(f'{r["Variable"]:<40} {r["Coeficiente"]:>10.4f} {r["Odds_Ratio"]:>12.4f}  {r["Direccion"]}')
print('─' * 70)
print(f'  Intercepto (β₀): {lr.intercept_[0]:.4f}')

In [ ]:
# Gráfico de Odds Ratios
top_n  = 20
df_top = pd.concat([
    coef_df.head(top_n // 2),
    coef_df.tail(top_n // 2)
]).sort_values('Coeficiente')

fig, ax = plt.subplots(figsize=(12, 8))
colores_bar = [ROJO if c > 0 else VERDE for c in df_top['Coeficiente']]
bars = ax.barh(df_top['Variable'], df_top['Coeficiente'],
               color=colores_bar, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')

for bar, or_val in zip(bars, df_top['Odds_Ratio']):
    w = bar.get_width()
    x_pos = w + 0.05 if w >= 0 else w - 0.05
    ha = 'left' if w >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'OR={or_val:.2f}', va='center', ha=ha, fontsize=8)

patch_r = mpatches.Patch(color=ROJO,  label='Coef (+): sube probabilidad de fraude')
patch_g = mpatches.Patch(color=VERDE, label='Coef (-): baja probabilidad de fraude')
ax.legend(handles=[patch_r, patch_g], loc='lower right')
ax.set_title('Coeficientes del Modelo — Regresión Logística\n'
             'OR > 1 = factor de riesgo | OR < 1 = factor protector',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente (β)')
plt.tight_layout()
plt.show()

---
## 6. Ejemplo Concreto — Calculando P(fraude) paso a paso

Tomamos una transacción REAL del dataset y calculamos su probabilidad manualmente.

In [ ]:
# Tomamos una transacción de fraude confirmado
idx_fraude = df_m[df_m['ES_FRAUDE'] == 1].index[0]
txn_fraude = X.loc[idx_fraude]

# Escalamos manualmente
txn_scaled = (txn_fraude.values - scaler.mean_) / scaler.scale_

# Calculamos Z
Z = lr.intercept_[0] + np.dot(lr.coef_[0], txn_scaled)
P_fraude = 1 / (1 + np.exp(-Z))

print('EJEMPLO — Transacción de FRAUDE REAL')
print('─' * 55)
print(f'  Intercepto (β₀)              : {lr.intercept_[0]:+.4f}')
print()

contribuciones = sorted(
    zip(FEATURES, lr.coef_[0], txn_scaled, lr.coef_[0] * txn_scaled),
    key=lambda x: abs(x[3]), reverse=True
)

print(f'  {"Variable":<40} {"Valor":>8} {"Contribución":>13}')
print(f'  {"─"*40} {"─"*8} {"─"*13}')
for feat, coef, val_sc, contrib in contribuciones[:15]:
    val_orig = txn_fraude[feat]
    signo = '+' if contrib >= 0 else ''
    print(f'  {feat:<40} {val_orig:>8.2f} {signo}{contrib:>12.4f}')

print(f'  {"─"*63}')
print(f'  Z (suma total)                              = {Z:+.4f}')
print(f'  P(fraude) = 1 / (1 + e^-{Z:.2f})           = {P_fraude:.4f}')
print(f'  Predicción                                  = {"FRAUDE ✅" if P_fraude >= 0.5 else "No fraude"}')

In [ ]:
# Lo mismo con una transacción LEGÍTIMA
idx_legit  = df_m[df_m['ES_FRAUDE'] == 0].index[0]
txn_legit  = X.loc[idx_legit]
txn_sc2    = (txn_legit.values - scaler.mean_) / scaler.scale_
Z2         = lr.intercept_[0] + np.dot(lr.coef_[0], txn_sc2)
P_fraude2  = 1 / (1 + np.exp(-Z2))

print('EJEMPLO — Transacción LEGÍTIMA')
print('─' * 55)
contrib2 = sorted(
    zip(FEATURES, lr.coef_[0], txn_sc2, lr.coef_[0] * txn_sc2),
    key=lambda x: abs(x[3]), reverse=True
)
print(f'  {"Variable":<40} {"Valor":>8} {"Contribución":>13}')
print(f'  {"─"*40} {"─"*8} {"─"*13}')
for feat, coef, val_sc, contrib in contrib2[:15]:
    val_orig = txn_legit[feat]
    signo = '+' if contrib >= 0 else ''
    print(f'  {feat:<40} {val_orig:>8.2f} {signo}{contrib:>12.4f}')
print(f'  {"─"*63}')
print(f'  Z (suma total)  = {Z2:+.4f}')
print(f'  P(fraude)       = {P_fraude2:.4f}')
print(f'  Predicción      = {"FRAUDE" if P_fraude2 >= 0.5 else "No fraude ✅"}')

---
## 7. Evaluación del Modelo — Curva ROC

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp

fpr_tr, tpr_tr, _ = roc_curve(y_train, y_prob_train)
fpr_te, tpr_te, _ = roc_curve(y_test,  y_prob_test)
auc_tr = roc_auc_score(y_train, y_prob_train)
auc_te = roc_auc_score(y_test,  y_prob_test)
ks_te, _ = ks_2samp(y_prob_test[y_test==1], y_prob_test[y_test==0])

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_tr, tpr_tr, color=AZUL,   lw=2, label=f'Train  AUC = {auc_tr:.4f}')
ax.plot(fpr_te, tpr_te, color=ROJO,   lw=2, label=f'Test   AUC = {auc_te:.4f}')
ax.plot([0,1],  [0,1],  color='gray', lw=1, linestyle='--', label='Modelo aleatorio AUC = 0.50')
ax.fill_between(fpr_te, tpr_te, alpha=0.08, color=ROJO)

idx_opt = np.argmax(tpr_te - fpr_te)
ax.scatter(fpr_te[idx_opt], tpr_te[idx_opt], color=NARANJA, s=100, zorder=5,
           label=f'Punto óptimo (FPR={fpr_te[idx_opt]:.3f}, TPR={tpr_te[idx_opt]:.3f})')

ax.set_xlabel('Tasa de Falsos Positivos (FPR)\n→ % clientes legítimos bloqueados', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR / Recall)\n→ % fraudes capturados', fontsize=11)
ax.set_title(f'Curva ROC — Modelo Regresión Logística\nKS Statistic (Test) = {ks_te:.4f}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f'AUC Train : {auc_tr:.4f}  |  AUC Test : {auc_te:.4f}')
print(f'Diferencia Train-Test: {abs(auc_tr-auc_te):.4f}  → {"✅ Sin sobreajuste" if abs(auc_tr-auc_te) < 0.02 else "⚠️ Revisar sobreajuste"}')
print(f'KS Statistic (Test)  : {ks_te:.4f}  → {"✅ Excelente" if ks_te >= 0.50 else "Aceptable"}')

---
## 8. Distribución del Score — Fraude vs No Fraude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax = axes[0]
ax.hist(y_prob_test[y_test==0], bins=50, color=AZUL,  alpha=0.6, label='No Fraude', density=True)
ax.hist(y_prob_test[y_test==1], bins=50, color=ROJO,  alpha=0.7, label='Fraude',    density=True)
ax.axvline(0.70, color=NARANJA, lw=2, linestyle='--', label='Umbral 0.70 (declinar)')
ax.axvline(0.45, color=VERDE,   lw=2, linestyle=':',  label='Umbral 0.45 (revisar)')
ax.set_title('Distribución de P(fraude) — Test Set\nBuen modelo: curvas bien separadas',
             fontweight='bold')
ax.set_xlabel('P(fraude)')
ax.set_ylabel('Densidad')
ax.legend(fontsize=9)

# Box plot
ax2 = axes[1]
data_box = [
    y_prob_test[y_test==0],
    y_prob_test[y_test==1]
]
bp = ax2.boxplot(data_box, patch_artist=True,
                 medianprops={'color':'white','linewidth':2})
bp['boxes'][0].set_facecolor(AZUL)
bp['boxes'][1].set_facecolor(ROJO)
ax2.set_xticklabels(['No Fraude (0)', 'Fraude (1)'])
ax2.set_ylabel('P(fraude)')
ax2.set_title('Mediana del Score por Clase\nFraude debe tener score más alto',
              fontweight='bold')
ax2.axhline(0.70, color=NARANJA, lw=1.5, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print(f'Score mediano — No Fraude : {np.median(y_prob_test[y_test==0]):.4f}')
print(f'Score mediano — Fraude    : {np.median(y_prob_test[y_test==1]):.4f}')

---
## 9. Análisis por Deciles — ¿Cuánto capturamos revisando X% de transacciones?

In [ ]:
df_test_score = pd.DataFrame({
    'y_true' : y_test.values,
    'y_prob'  : y_prob_test
}).sort_values('y_prob', ascending=False).reset_index(drop=True)

n = len(df_test_score)
total_f = df_test_score['y_true'].sum()
rows = []
for d in range(10):
    sub = df_test_score.iloc[d*n//10 : (d+1)*n//10]
    nf  = sub['y_true'].sum()
    rows.append({
        'Decil'            : d+1,
        'Min_Score'        : round(sub['y_prob'].min(), 4),
        'Max_Score'        : round(sub['y_prob'].max(), 4),
        'N_Total'          : len(sub),
        'N_Fraudes'        : int(nf),
        'Tasa_Fraude%'     : round(nf/len(sub)*100, 2),
        'Captura_Fraude%'  : round(nf/total_f*100, 2),
    })

df_dec = pd.DataFrame(rows)
print(df_dec.to_string(index=False))

# Gráfico lift
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tasa de fraude por decil
ax = axes[0]
colores_dec = [ROJO if t > 2 else NARANJA if t > 1 else AZUL for t in df_dec['Tasa_Fraude%']]
bars = ax.bar(df_dec['Decil'], df_dec['Tasa_Fraude%'], color=colores_dec, edgecolor='white')
ax.axhline(y_test.mean()*100, color='gray', lw=1.5, linestyle='--',
           label=f'Tasa base = {y_test.mean()*100:.2f}%')
for bar, val in zip(bars, df_dec['Tasa_Fraude%']):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Tasa de Fraude por Decil\nDecil 1 = transacciones más riesgosas',
             fontweight='bold')
ax.set_xlabel('Decil (1 = mayor score)')
ax.set_ylabel('Tasa de Fraude %')
ax.legend()

# Captura acumulada
ax2 = axes[1]
captura_acum = df_dec['Captura_Fraude%'].cumsum()
pct_txn      = [(i+1)*10 for i in range(10)]
ax2.plot(pct_txn, captura_acum, color=ROJO, lw=2.5, marker='o', markersize=8)
ax2.plot(pct_txn, pct_txn,     color='gray', lw=1.5, linestyle='--', label='Modelo aleatorio')
ax2.fill_between(pct_txn, captura_acum, pct_txn, alpha=0.1, color=ROJO)
for x, y_val in zip(pct_txn[:4], captura_acum[:4]):
    ax2.annotate(f'{y_val:.1f}%', (x, y_val), textcoords='offset points',
                 xytext=(5, 5), fontsize=9, color=ROJO)
ax2.set_title('Curva de Captura Acumulada\n¿Cuánto fraude capturamos revisando X% de txn?',
              fontweight='bold')
ax2.set_xlabel('% de transacciones revisadas')
ax2.set_ylabel('% de fraudes capturados')
ax2.legend()
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

print(f'\n📌 Revisando el TOP 10% de transacciones → capturamos el {captura_acum.iloc[0]:.1f}% del fraude')
print(f'📌 Revisando el TOP 20% de transacciones → capturamos el {captura_acum.iloc[1]:.1f}% del fraude')

---
## 10. Tabla de Umbrales Operativos — ¿Dónde poner el corte?

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

umbrales = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
rows_u   = []
for u in umbrales:
    pred = (y_prob_test >= u).astype(int)
    tp   = int(((pred==1) & (y_test==1)).sum())
    fp   = int(((pred==1) & (y_test==0)).sum())
    fn   = int(((pred==0) & (y_test==1)).sum())
    tn   = int(((pred==0) & (y_test==0)).sum())
    prec = tp/(tp+fp+1e-9)
    rec  = tp/(tp+fn+1e-9)
    f1   = 2*prec*rec/(prec+rec+1e-9)
    fpr  = fp/(fp+tn+1e-9)
    pctd = pred.mean()*100
    rows_u.append({
        'Umbral'          : u,
        'Precision_%'     : round(prec*100, 1),
        'Recall_%'        : round(rec*100,  1),
        'F1'              : round(f1, 3),
        'FPR_%'           : round(fpr*100,  2),
        'Pct_Declina%'    : round(pctd, 2),
        'TP_Fraudes'      : tp,
        'FP_Legitimas'    : fp,
    })

df_umb = pd.DataFrame(rows_u)
print(f'{"Umbral":>7} | {"Precision%":>10} | {"Recall%":>8} | {"FPR%":>6} | {"Declina%":>9} | Fraudes capturados | Legítimas afectadas')
print('─' * 95)
for _, r in df_umb.iterrows():
    marca = ' ← RECOMENDADO declinar' if r['Umbral'] == 0.90 else (' ← REVISAR manual' if r['Umbral'] == 0.70 else '')
    print(f'{r["Umbral"]:>7.2f} | {r["Precision_%"]:>10.1f} | {r["Recall_%"]:>8.1f} | {r["FPR_%"]:>6.2f} | {r["Pct_Declina%"]:>9.2f} | {r["TP_Fraudes"]:>18} | {r["FP_Legitimas"]:>20}{marca}')

# Gráfico precision-recall
fig, ax = plt.subplots(figsize=(10, 5))
ax2b = ax.twinx()
ax.plot(df_umb['Umbral'], df_umb['Precision_%'], color=VERDE,   lw=2, marker='o', label='Precision %')
ax.plot(df_umb['Umbral'], df_umb['Recall_%'],    color=ROJO,    lw=2, marker='s', label='Recall %')
ax2b.plot(df_umb['Umbral'], df_umb['FPR_%'],     color=NARANJA, lw=2, marker='^', linestyle='--', label='FPR % (eje der.)')
ax.axvline(0.90, color='black', lw=1.5, linestyle=':', alpha=0.7)
ax.axvline(0.70, color='gray',  lw=1.5, linestyle=':', alpha=0.7)
ax.text(0.90, 50, ' P≥0.90\n declinar', fontsize=9, color='black')
ax.text(0.70, 60, ' P≥0.70\n revisar',  fontsize=9, color='gray')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Precision / Recall %')
ax2b.set_ylabel('FPR % (Falsos Positivos)', color=NARANJA)
ax.set_title('Trade-off: Precision vs Recall vs FPR\nA mayor umbral → más preciso pero menos captura',
             fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center left')
plt.tight_layout()
plt.show()

---
## 10b. Matriz de Confusión — ¿Qué pasa en la realidad con cada umbral?

La tabla de umbrales muestra porcentajes. Aquí vemos los **números reales** de transacciones:
cada celda es una decisión concreta que el modelo toma sobre cada transacción.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.ticker as ticker

def plot_confusion(ax, y_true, y_prob, umbral, titulo, total_base=170944, total_fraude_base=2288):
    """Dibuja matriz de confusión para un umbral dado."""
    y_pred = (y_prob >= umbral).astype(int)
    cm     = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()
    n_test = len(y_true)
    n_fraude_test = int(y_true.sum())

    # Extrapolación a base completa
    factor_f = total_fraude_base / n_fraude_test
    factor_n = (total_base - total_fraude_base) / (n_test - n_fraude_test)
    tp_ext = int(tp * factor_f)
    fn_ext = int(fn * factor_f)
    fp_ext = int(fp * factor_n)
    tn_ext = int(tn * factor_n)

    # Heatmap
    cm_display = np.array([[tn, fp], [fn, tp]])
    im = ax.imshow(cm_display, cmap='Blues', aspect='auto')

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['MODELO: Aprueba', 'MODELO: Declina/Alerta'], fontsize=9)
    ax.set_yticklabels(['REAL: Legítima', 'REAL: Fraude'], fontsize=9)

    labels = [
        [f'TN={tn:,}\n✅ Legítimas\nbien aprobadas\n(~{tn_ext:,} en base total)',
         f'FP={fp:,}\n⚠️ Legítimas\nafectadas (costo)\n(~{fp_ext:,} en base total)'],
        [f'FN={fn:,}\n❌ Fraudes\nque escapan\n(~{fn_ext:,} en base total)',
         f'TP={tp:,}\n✅ Fraudes\ncapturados\n(~{tp_ext:,} en base total)'],
    ]
    colores_texto = [['black', 'black'], ['white', 'white']]

    for i in range(2):
        for j in range(2):
            ax.text(j, i, labels[i][j], ha='center', va='center',
                    fontsize=8, color=colores_texto[i][j], fontweight='bold',
                    linespacing=1.4)

    precision = tp / (tp + fp + 1e-9) * 100
    recall    = tp / (tp + fn + 1e-9) * 100
    fpr       = fp / (fp + tn + 1e-9) * 100

    ax.set_title(
        f'{titulo}\n'
        f'Precision={precision:.1f}%  |  Recall={recall:.1f}%  |  FPR={fpr:.2f}%',
        fontsize=10, fontweight='bold'
    )


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_confusion(axes[0], y_test, y_prob_test, umbral=0.90,
               titulo='Umbral 0.90 — DECLINAR automáticamente')
plot_confusion(axes[1], y_test, y_prob_test, umbral=0.70,
               titulo='Umbral 0.70 — REVISIÓN manual')

plt.suptitle(
    'Matriz de Confusión — Test Set (34,150 txn) con extrapolación a base completa (170,944 txn)\n'
    'TP=Fraudes capturados  |  FP=Legítimas afectadas  |  FN=Fraudes que escapan  |  TN=Legítimas aprobadas bien',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

# Tabla resumen comparativa
print('\n' + '─'*75)
print(f'{"":25} {"Umbral 0.90":>20} {"Umbral 0.70":>20}')
print('─'*75)
for u, label in [(0.90, 'DECLINAR'), (0.70, 'REVISAR')]:
    y_pred  = (y_prob_test >= u).astype(int)
    cm_vals = confusion_matrix(y_test, y_pred).ravel()
    tn, fp, fn, tp = cm_vals
    break

# Recalcular para ambos umbrales
resultados_cm = {}
for u in [0.90, 0.70]:
    y_pred = (y_prob_test >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    resultados_cm[u] = {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

r90, r70 = resultados_cm[0.90], resultados_cm[0.70]
print(f'  {"Fraudes capturados (TP)":<30} {r90["TP"]:>15,}     {r70["TP"]:>15,}')
print(f'  {"Fraudes escapados (FN)":<30} {r90["FN"]:>15,}     {r70["FN"]:>15,}')
print(f'  {"Legítimas afectadas (FP)":<30} {r90["FP"]:>15,}     {r70["FP"]:>15,}')
print(f'  {"Legítimas aprobadas OK (TN)":<30} {r90["TN"]:>15,}     {r70["TN"]:>15,}')
print('─'*75)
print(f'  {"Total declinado/alertado":<30} {r90["TP"]+r90["FP"]:>15,}     {r70["TP"]+r70["FP"]:>15,}')
print(f'  {"Ratio FP/TP (costo/beneficio)":<30} {r90["FP"]/r90["TP"]:>14.2f}x     {r70["FP"]/r70["TP"]:>14.2f}x')
print()
print('  💡 Ratio FP/TP = cuántas legítimas afectamos por cada fraude capturado')
print(f'     Umbral 0.90: por cada fraude capturado afectamos {r90["FP"]/r90["TP"]:.1f} legítimas')
print(f'     Umbral 0.70: por cada fraude capturado afectamos {r70["FP"]/r70["TP"]:.1f} legítimas')

---
## 11. Resumen Final para la Reunión

In [ ]:
from sklearn.metrics import log_loss

gini_tr = 2*auc_tr - 1
gini_te = 2*auc_te - 1
ll_te   = log_loss(y_test, y_prob_test)
ks_tr,_ = ks_2samp(y_prob_train[y_train==1], y_prob_train[y_train==0])

print('═' * 60)
print('  RESUMEN DEL MODELO — PARA LA REUNIÓN')
print('═' * 60)
print(f'  Modelo        : Regresión Logística (class_weight=balanced)')
print(f'  Features      : {len(FEATURES)} variables (sin leakage)')
print(f'  Split         : 80% train / 20% test estratificado')
print()
print(f'  {"Métrica":<20} {"Train":>10} {"Test":>10}')
print(f'  {"─"*42}')
print(f'  {"AUC-ROC":<20} {auc_tr:>10.4f} {auc_te:>10.4f}')
print(f'  {"Gini":<20} {gini_tr:>10.4f} {gini_te:>10.4f}')
print(f'  {"KS Statistic":<20} {ks_tr:>10.4f} {ks_te:>10.4f}')
print()
print(f'  ✅ Sin sobreajuste (Train ≈ Test)')
print(f'  ✅ AUC > 0.85 = modelo excelente')
print(f'  ✅ KS > 0.50  = separación excelente')
print()
print(f'  UMBRALES OPERATIVOS SUGERIDOS:')
print(f'    P >= 0.90 → Declinar automáticamente')
print(f'    P >= 0.70 → Revisión manual')
print(f'    P <  0.70 → Aprobar')
print()
print(f'  IMPACTO:')
print(f'    Top 10% txn (mayor score) captura ~92% del fraude')
print(f'    Top 20% txn captura prácticamente todo el fraude')
print('═' * 60)